[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S15/S15_01_seguimiento_experimentos.ipynb)

# S15 · 01 — Seguimiento de experimentos y reproducibilidad

**Tiempo estimado:** 1 h 15 min · **Sprint 15 (Proyecto: modelado y experimentos)** · píldora de apoyo

**Contexto.** En el proyecto final vuestro equipo probará muchos modelos y muchos ajustes. Al tercer día
nadie recuerda *"¿aquel 0,78 era con la semilla 1 o la 42? ¿con o sin la variable de antelación?"*. La
rúbrica (criterio C) premia el **registro de experimentos**, y un modelo que no se puede reproducir no se
puede defender. Aquí montamos un sistema de registro **muy simple** (un CSV) que funciona en cualquier
equipo. Usamos las reservas de TurisData como datos de práctica; en el proyecto usaréis los vuestros.

**Al terminar sabrás:**
1. Fijar semillas y anotar versiones para que otra persona obtenga *exactamente* tus resultados.
2. Registrar cada experimento (modelo, hiperparámetros, métrica, tiempo, datos) en un CSV compartido en Git.
3. Comparar modelos con **validación cruzada** y elegir el candidato **sin mirar el test**.
4. Detectar una **fuga de datos** por una métrica "demasiado buena".
5. (Opcional) Usar MLflow en local como alternativa más completa.

In [ ]:
# Preparación (ejecuta esta celda siempre la primera)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta


In [ ]:
import os, sys, time, json, random, hashlib
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

sns.set_theme(style="whitegrid")

## 1. Reproducibilidad: semillas y versiones

**Idea.** Muchos pasos usan azar (separar datos, inicializar pesos, submuestrear). Si no fijas la
**semilla**, cada ejecución da un resultado ligeramente distinto. Además, distintas **versiones** de las
librerías pueden cambiar resultados. Regla del equipo: **una función `fijar_semillas` al inicio de cada
notebook** y las versiones anotadas en el registro.

### Ejercicio 1
Escribe `fijar_semillas(semilla)` que fije la semilla de `random` y de `numpy` (`np.random.seed`),
guarde `PYTHONHASHSEED` en `os.environ` (como texto) y **devuelva** un generador
`np.random.default_rng(semilla)`. (Con TensorFlow añadirías `tf.random.set_seed(semilla)`.)

In [ ]:
def fijar_semillas(semilla):
    # TODO: Cada biblioteca con azar tiene su propia semilla; la variable de entorno debe ser un str; default_rng crea el generador moderno de numpy
    ...

SEMILLA = 42
rng = fijar_semillas(SEMILLA)
a = (random.random(), np.random.rand(), rng.random())
rng = fijar_semillas(SEMILLA)
b = (random.random(), np.random.rand(), rng.random())
assert a == b, "Al llamar dos veces a fijar_semillas(42) deberías obtener la misma secuencia aleatoria en random, np.random y el rng devuelto"
assert os.environ.get("PYTHONHASHSEED") == "42", "PYTHONHASHSEED debe guardarse como texto '42'"
print("Ejercicio 1 correcto ✔")

In [ ]:
# Ficha del entorno: cópiala en la memoria técnica y en el README del equipo
ENTORNO = {"python": sys.version.split()[0], "numpy": np.__version__, "pandas": pd.__version__, "scikit-learn": sklearn.__version__}
print(json.dumps(ENTORNO, indent=2))

## 2. El registro de experimentos: una fila por prueba

**Idea.** Cada vez que entrenas y evalúas, anotas una fila: **cuándo, qué modelo, con qué ajustes, con
qué datos, qué métrica y cuánto tardó**. Un CSV en el repositorio del equipo (`experimentos/registro.csv`)
se puede abrir con Excel, se versiona con Git y no requiere instalar nada.

Columnas mínimas: `id`, `fecha`, `responsable`, `modelo`, `hiperparametros` (JSON), `datos_hash`,
`semilla`, `metrica`, `valor_cv`, `desv_cv`, `tiempo_s`, `notas`.

### Ejercicio 2
Completa `registrar(...)`, que añade **una fila** al CSV `RUTA_REGISTRO`: si el archivo no existe, lo crea
con la cabecera; si existe, añade la fila sin repetir la cabecera. El `id` es el número de filas ya
existentes + 1. Devuelve el `id` asignado.

In [ ]:
RUTA_REGISTRO = "experimentos/registro.csv"
COLUMNAS = ["id", "fecha", "responsable", "modelo", "hiperparametros", "datos_hash", "semilla",
            "metrica", "valor_cv", "desv_cv", "tiempo_s", "notas"]
os.makedirs("experimentos", exist_ok=True)
if os.path.exists(RUTA_REGISTRO):
    os.remove(RUTA_REGISTRO)          # empezamos el laboratorio con el registro vacío

def registrar(responsable, modelo, hiperparametros, datos_hash, semilla, metrica, valor_cv, desv_cv, tiempo_s, notas=""):
    # TODO: Cuenta las filas con pd.read_csv si el archivo existe; crea un DataFrame de una fila y guárdalo con to_csv en modo "a" (header solo si es nuevo)
    ...

id1 = registrar("prueba", "ModeloDePrueba", {"a": 1}, "abc123", 42, "AUC", 0.5, 0.01, 0.1, "fila de prueba")
id2 = registrar("prueba", "ModeloDePrueba", {"a": 2}, "abc123", 42, "AUC", 0.6, 0.01, 0.1)
tabla_prueba = pd.read_csv(RUTA_REGISTRO)
assert (id1, id2) == (1, 2), "Los id deben ser 1, 2, ... (filas existentes + 1)"
assert list(tabla_prueba.columns) == COLUMNAS and len(tabla_prueba) == 2, "El CSV debe tener la cabecera una sola vez y 2 filas"
os.remove(RUTA_REGISTRO)              # borramos las filas de prueba
print("Ejercicio 2 correcto ✔")

## 3. Experimentos: baseline y modelos con validación cruzada

**Idea.** Para comparar modelos se usa **validación cruzada**: se parte el conjunto de entrenamiento en 5
trozos y se entrena 5 veces, evaluando cada vez en el trozo que no se ha visto. Obtenemos una **media**
y una **desviación** (¡importante para saber si una diferencia es real o ruido!). El **test** se
reserva para el final y se usa **una sola vez**. Con series temporales, la partición debe respetar el orden
del tiempo (`TimeSeriesSplit`), nunca al azar.

**Ejemplo resuelto:** preparamos datos y una función que entrena, evalúa y **registra automáticamente**.

In [ ]:
reservas = pd.read_csv(dato("reservas_turisdata.csv"), parse_dates=["fecha_reserva", "fecha_llegada"])
reservas["mes_llegada"] = reservas["fecha_llegada"].dt.month
X = reservas.drop(columns=["id_reserva", "fecha_reserva", "fecha_llegada", "cancelada"])
y = reservas["cancelada"]
X_ent, X_test, y_ent, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEMILLA)

# huella de los datos: si alguien cambia el CSV, el hash cambia y lo sabremos
DATOS_HASH = hashlib.md5(pd.util.hash_pandas_object(reservas, index=True).values.tobytes()).hexdigest()[:8]
print("Huella de los datos:", DATOS_HASH, "· entrenamiento:", X_ent.shape, "· test (sin tocar):", X_test.shape)

categoricas = X.select_dtypes("object").columns.tolist()
def crear_pipeline(estimador, columnas):
    numericas = [c for c in columnas if c not in categoricas]
    prep = ColumnTransformer([("num", StandardScaler(), numericas), ("cat", OneHotEncoder(handle_unknown="ignore"), categoricas)])
    return Pipeline([("prep", prep), ("m", estimador)])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)

def experimento(nombre, estimador, hiperparametros, responsable="equipo", notas="", Xd=None):
    """Evalúa con CV (AUC), mide el tiempo y registra la fila. Devuelve (media, desviación)."""
    Xd = X_ent if Xd is None else Xd
    t0 = time.time()
    puntuaciones = cross_val_score(crear_pipeline(estimador, Xd.columns), Xd, y_ent, cv=cv, scoring="roc_auc")
    registrar(responsable, nombre, hiperparametros, DATOS_HASH, SEMILLA, "AUC", puntuaciones.mean(), puntuaciones.std(), time.time() - t0, notas)
    return puntuaciones.mean(), puntuaciones.std()

print("Baseline (predice la clase más frecuente):", experimento("Baseline", DummyClassifier(strategy="prior"), {"strategy": "prior"}, notas="referencia mínima"))
print("Regresión logística:", experimento("RegresionLogistica", LogisticRegression(max_iter=2000), {"C": 1.0}))

### Ejercicio 3
Registra dos experimentos más con la función `experimento(nombre, estimador, hiperparametros)`:
un **bosque aleatorio** `RandomForestClassifier(n_estimators=200, min_samples_leaf=5, random_state=SEMILLA, n_jobs=2)`
con nombre `"RandomForest"` y un **gradient boosting** `HistGradientBoostingClassifier(random_state=SEMILLA)` con
nombre `"HistGradientBoosting"` (hiperparámetros: los que uses, en un diccionario). Después lee el registro
en `tabla` (`pd.read_csv(RUTA_REGISTRO)`) y guarda en `candidato` el **nombre del modelo con mayor `valor_cv`**.

In [ ]:
# TODO: Llama a experimento dos veces; lee el CSV con RUTA_REGISTRO; idxmax sobre valor_cv te da la fila del mejor
...

assert len(tabla) == 4, "El registro debería tener 4 filas: baseline, logística, bosque y gradient boosting"
assert set(["RandomForest", "HistGradientBoosting"]) <= set(tabla["modelo"]), "Registra los dos modelos con los nombres indicados"
assert candidato == tabla.sort_values("valor_cv").iloc[-1]["modelo"], "candidato debe ser el modelo con mayor valor_cv"
print("Ejercicio 3 correcto ✔ — candidato provisional:", candidato)
tabla[["id", "modelo", "hiperparametros", "valor_cv", "desv_cv", "tiempo_s"]]

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
t = tabla.sort_values("valor_cv")
ax.barh(t["modelo"], t["valor_cv"], xerr=t["desv_cv"], color="tab:blue", capsize=4)
ax.axvline(0.5, color="gray", linestyle="--", label="Azar (AUC 0,5)")
ax.set_xlim(0.4, 0.85); ax.set_xlabel("AUC medio en validación cruzada (± desviación)")
ax.set_title("Comparativa de modelos (validación cruzada, sin usar el test)"); ax.legend(loc="lower right")
plt.tight_layout(); plt.show()

**Lectura importante:** si dos modelos difieren menos que su desviación, **no hay diferencia real**.
En ese caso elige el más simple, más rápido o más fácil de explicar. Anótalo en la columna `notas`.

## 4. Ajuste de hiperparámetros, también registrado

### Ejercicio 4
Prueba un bosque aleatorio con `max_depth` = 3, 6 y 10 (resto de parámetros como antes), **registrando cada
prueba** con `experimento` y el nombre `f"RandomForest_prof{d}"`. Guarda en `resultados_prof` un diccionario
`{profundidad: AUC medio}` y en `mejor_prof` la profundidad con mejor AUC.

In [ ]:
# TODO: Un bucle sobre las tres profundidades; experimento devuelve (media, desviación)
...

assert set(resultados_prof) == {3, 6, 10}, "resultados_prof debe tener las claves 3, 6 y 10"
assert len(pd.read_csv(RUTA_REGISTRO)) == 7, "Deberías tener 7 filas en el registro (4 anteriores + 3 nuevas)"
assert mejor_prof in (3, 6, 10)
print("Ejercicio 4 correcto ✔ — mejor profundidad:", mejor_prof, {k: round(v, 3) for k, v in resultados_prof.items()})

## 5. Alarma: una métrica demasiado buena (fuga de datos)

**Idea.** Si tu modelo obtiene un resultado espectacular (AUC 0,99 en un problema difícil), **sospecha**.
Casi siempre hay **fuga de datos**: una variable que incluye información del futuro o del propio
objetivo. Ejemplo: una columna `reembolso_emitido` solo existe **después** de cancelar. El modelo
"acierta" pero es inútil en producción, porque en el momento de decidir esa columna aún no existe.

**Ejemplo resuelto:** la registramos con una nota de alarma y la comparamos.

In [ ]:
X_fuga = X_ent.copy()
X_fuga["reembolso_emitido"] = ((y_ent == 1) & (rng.random(len(y_ent)) < 0.9)).astype(int)
m, d = experimento("LogisticaConFuga", LogisticRegression(max_iter=2000), {"C": 1.0}, notas="ALARMA: AUC sospechosamente alto (fuga: reembolso_emitido)", Xd=X_fuga)
print(f"AUC con la variable con fuga: {m:.3f} (vs. ≈ 0,74 sin ella) → ¡descártala y anótalo en el registro!")

**Cómo prevenir fugas:** (1) para cada variable pregunta *"¿la conocería en el momento de decidir?"*;
(2) haz el **preprocesado dentro de un `Pipeline`** (así el escalado se aprende solo con el entrenamiento);
(3) en series, parte por fechas; (4) sospecha de lo "demasiado bueno".

## 6. Evaluar el candidato en el test, una sola vez

### Ejercicio 5
Elige el candidato de la tabla (**solo con criterios de validación cruzada, sin fuga**): el
`RandomForest_prof{mejor_prof}` o el ganador del ejercicio 3, el que tenga mayor `valor_cv` entre las filas
**sin** la palabra "Fuga" en el nombre. Reentrénalo con todo `X_ent` y calcula el AUC en `X_test`
(`auc_test`). Guarda en `auc_cv_candidato` su AUC de validación cruzada para compararlos.
*(Pista: si el test sale muy por debajo de la validación cruzada, hay sobreajuste o fuga.)*

In [ ]:
# TODO: Filtra las filas de la tabla sin "Fuga"; según el nombre elegido, construye el estimador con los mismos hiperparámetros; predict_proba(X_test)[:, 1]
...

assert 0.6 < auc_test < 0.85, "El AUC en test debería ser realista (entre 0,6 y 0,85); revisa que hayas excluido el modelo con fuga"
assert abs(auc_test - auc_cv_candidato) < 0.06, "Test y validación cruzada deberían ser parecidos; si no, sospecha de fuga o sobreajuste"
print(f"Ejercicio 5 correcto ✔ — candidato: {fila['modelo']} · AUC validación {auc_cv_candidato:.3f} · AUC test {auc_test:.3f}")

## 7. (Opcional) MLflow en local

**MLflow** es una herramienta profesional de seguimiento: guarda parámetros, métricas y modelos y ofrece
una interfaz web (`mlflow ui`). **No es obligatoria** ni necesaria para el proyecto: el CSV cumple con
la rúbrica. Si tu equipo quiere probarla, pon `USAR_MLFLOW = True`. Necesita `%pip install -q mlflow`
(descarga, requiere internet) y **no se ejecuta por defecto**.

In [ ]:
USAR_MLFLOW = False        # OPCIONAL: cámbialo a True solo si tu equipo quiere probar MLflow (necesita internet para instalarlo)

if USAR_MLFLOW:
    # %pip install -q mlflow       # <- descomenta en Colab; en local: pip install mlflow
    import mlflow
    try:
        import sqlalchemy                                  # instalado con MLflow completo: usamos la base de datos local
        URI = "sqlite:///mlflow.db"
    except ImportError:                                    # instalación ligera: carpeta local mlruns/
        os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
        URI = "file:./mlruns"
    mlflow.set_tracking_uri(URI)
    mlflow.set_experiment("turisdata_cancelaciones")
    with mlflow.start_run(run_name="RandomForest_demo"):
        mlflow.log_params({"n_estimators": 200, "min_samples_leaf": 5, "semilla": SEMILLA})
        mlflow.log_metric("auc_cv", float(fila["valor_cv"]))
        mlflow.log_metric("auc_test", float(auc_test))
        mlflow.set_tag("datos_hash", DATOS_HASH)
    print(f"Registrado en {URI}. Para verlo, en una terminal:  mlflow ui --backend-store-uri {URI}  y visita http://127.0.0.1:5000")
else:
    print("MLflow desactivado (opcional). El registro en CSV ya cumple lo que pide la rúbrica.")

## Mini-reto integrador
Aplica esto a **tu proyecto**: crea la carpeta `experimentos/` en el repositorio del equipo, copia
`fijar_semillas`, `registrar` y `experimento` a un archivo `utilidades.py`, registra ≥ 3 modelos + baseline
con tus datos y haz un *commit* con el CSV. Cada integrante debe tener **al menos una fila** con su nombre
en `responsable` (sirve como evidencia de contribución).

## Preguntas de reflexión
1. ¿Qué dos cosas necesita otra persona para reproducir exactamente tu mejor resultado?
2. Dos modelos tienen AUC 0,74 ± 0,02 y 0,75 ± 0,02. ¿Puedes afirmar que el segundo es mejor? ¿Qué elegirías?
3. ¿Por qué no se debe elegir el modelo mirando el resultado en el test? ¿Qué pasa si se hace varias veces?
4. Tu equipo obtiene AUC 0,97 en la primera prueba. Enumera tres cosas que revisarías antes de celebrarlo.

## Qué has aprendido
- Fijar semillas, anotar versiones y huella de datos.
- Registrar experimentos en un CSV versionado en Git.
- Comparar con validación cruzada (media ± desviación) y reservar el test para el final.
- Reconocer y prevenir fugas de datos.